# Lesson 09: Building Semantic Search Applications

In this notebook, you will:
1. Load a self-contained video transcript segment index (speaker, 3-minute bucket, 60-word summary).
2. Generate embeddings with **Google AI Studio (`gemini-embedding-001`)** or open-source **`sentence-transformers`**.
3. Rank results by cosine similarity (>= 0.35) and output timestamped `https://youtu.be/{videoId}?t={seconds}` deep links.

In [ ]:
!pip install -q -U google-genai sentence-transformers pandas numpy

In [ ]:
import os, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

segments = [
    {"videoId": "vid101", "seconds": 180, "speaker": "Seth", "title": "Vector Embeddings Explained", "summary": "Dense vector embeddings map sentences into geometric space so cosine similarity finds semantically related video moments."},
    {"videoId": "vid102", "seconds": 360, "speaker": "Bethany", "title": "Chunking with Overlap", "summary": "Splitting transcripts into 3-minute segments with a 20-word overlap prevents losing context across boundaries."},
    {"videoId": "vid103", "seconds": 540, "speaker": "Korey", "title": "Prompt Engineering Basics", "summary": "Use delimiters, system instructions, and few-shot examples to reduce LLM fabrications."},
]
df = pd.DataFrame(segments)
embedder = SentenceTransformer("all-MiniLM-L6-v2")
df["embedding"] = list(embedder.encode(df["summary"].tolist()))

In [ ]:
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def get_videos(query, dataset, rows=5, threshold=0.35):
    q_vec = embedder.encode([query])[0]
    out = dataset.copy()
    out["similarity"] = out["embedding"].apply(lambda v: cosine_similarity(q_vec, v))
    return out[out["similarity"] >= threshold].sort_values("similarity", ascending=False).head(rows)

results = get_videos("how do I measure similarity between transcript chunks?", df)
for _, r in results.iterrows():
    print(f"{r['title']} ({r['speaker']}) | sim={r['similarity']:.3f} -> https://youtu.be/{r['videoId']}?t={r['seconds']}")